# MASI Colab Smoke Test (Fresh Clone)

This notebook is the safest Colab bootstrap path for MASI smoke testing.

It does all of the following:
- deletes any old `/content/MASI` clone,
- clones the latest repo again,
- installs dependencies into a fresh local virtual environment,
- downloads the bounded CSJ review slice,
- runs the one-click smoke pipeline,
- checks whether the expected output manifest exists before trying to open it.

The notebook itself does not contain MASI training logic. It only orchestrates the checked-in repository scripts so the Colab run stays consistent with the repo.

## Configure Repo

Set your repo URL and branch below before running all cells.

In [ ]:
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
BRANCH = "main"
REPO_DIR = "/content/MASI"
STORAGE_ROOT = "/content/masi_storage"

## Fresh Clone

This cell deletes any older Colab checkout so you always run against the latest pushed repo state.

In [ ]:
%cd /content
!rm -rf "$REPO_DIR"
!git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!git rev-parse --short HEAD
!ls -lah scripts/train_masi.py configs/masi_train_csj_smoke.json notebooks/03_colab_smoke_test.ipynb

## Install Dependencies

In [ ]:
%cd "$REPO_DIR"
!python -m venv .venv
!.venv/bin/pip install --upgrade pip
!.venv/bin/pip install -e ".[recommender]"

## Prepare Folders

In [ ]:
%cd "$REPO_DIR"
!mkdir -p data/raw/amazon_reviews_2023
!mkdir -p "$STORAGE_ROOT"

## Download Bounded CSJ Slice

This is the smoke-test data path, not the full benchmark download.

In [ ]:
%cd "$REPO_DIR"
!PYTHONPATH=src python scripts/download_amazon_csj_dataset.py
!ls -lh data/raw/amazon_reviews_2023

## Run Smoke Pipeline

In [ ]:
%cd "$REPO_DIR"
!PYTHONPATH=src .venv/bin/python scripts/train_masi.py \
  --config configs/masi_train_csj_smoke.json \
  --storage-root "$STORAGE_ROOT"

## Inspect Output Files

In [ ]:
!find "$STORAGE_ROOT" -maxdepth 4 -type f | sort

## Load Manifest Safely

This cell checks whether the smoke run actually produced the manifest before trying to open it.

In [ ]:
import json
from pathlib import Path

manifest_path = Path(STORAGE_ROOT) / "outputs" / "amazon_csj_smoke_train" / "run_manifest.json"
print("Run manifest path:", manifest_path)
print("Exists:", manifest_path.exists())

if manifest_path.exists():
    with manifest_path.open("r", encoding="utf-8") as handle:
        manifest = json.load(handle)
    print("Environment:", manifest["environment"])
    print("Token summary path:", manifest["token_summary_path"])
    print("Experiment summary path:", manifest["experiment_summary_path"])
else:
    print("Smoke run did not finish successfully. Re-check the previous training cell output.")

## Load Experiment Summary Safely

In [ ]:
import json
from pathlib import Path

summary_path = Path(STORAGE_ROOT) / "outputs" / "amazon_csj_smoke_train" / "phase3_experiment" / "experiment_summary.json"
print("Summary path:", summary_path)
print("Exists:", summary_path.exists())

if summary_path.exists():
    with summary_path.open("r", encoding="utf-8") as handle:
        summary = json.load(handle)
    high_signal = {
        "mlm_status": summary.get("mlm_status"),
        "generative_finetuning_status": summary.get("generative_finetuning_status"),
        "num_items": summary.get("num_items"),
        "num_train_examples": summary.get("num_train_examples"),
        "num_mlm_examples": summary.get("num_mlm_examples"),
        "warm_metrics": summary.get("warm_metrics"),
        "cold_metrics": summary.get("cold_metrics"),
    }
    print(json.dumps(high_signal, indent=2))
else:
    print("Experiment summary is missing because the smoke run did not complete.")